In [9]:
import time
import cv2
import mediapipe as mp
import numpy as np
cap = cv2.VideoCapture(0)
h,w=cap.get(cv2.CAP_PROP_FRAME_HEIGHT),cap.get(cv2.CAP_PROP_FRAME_WIDTH)
mp_drawing = mp.solutions.drawing_utils
mp_face = mp.solutions.face_detection
face_detection = mp_face.FaceDetection()
start_time=time.time()
fps_start = time.time()
fps=0
frame_count = 0
max_time=10
while True:
  ret, frame = cap.read()
  if not ret:
    break
  frame_count+=1

  h,w,c=frame.shape
  frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  frame_rgb.flags.writeable = False
  results = face_detection.process(frame_rgb)
  frame_rgb.flags.writeable = True
  frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
  if results.detections:
    spend_time=time.time()-start_time
    cv2.putText(frame, f"Spended Time in sec: {spend_time:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    if spend_time>max_time:
      cv2.rectangle(frame, (0, 0), (int(w), int(h)), (0, 0, 255), 5) 
      cv2.putText(frame, "Time Out", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
      cv2.setWindowProperty("Security Alert", cv2.WND_PROP_TOPMOST, 1)
    #   cv2.setWindowProperty("Security Alert", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
      # cv2.setWindowProperty("Security Alert", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_NORMAL)
      
    for detection in results.detections:
      mp_drawing.draw_detection(frame, detection)
  else:
    print("No face detected")
  end_time=time.time()
  if end_time - fps_start >= 1.0:
    fps=frame_count/(end_time-fps_start)
    fps_start=end_time
    frame_count=0
  cv2.putText(frame, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
  cv2.imshow('Security Alert', frame)
  if cv2.waitKey(1) & 0xFF == ord('q'):
    break
cap.release()
cv2.destroyAllWindows()
   


# # Time Tracker with OpenCV and MediaPipe

In [1]:

import time
import cv2
import mediapipe as mp

# ==================== CONFIGURATION ====================

CAMERA_INDEX = 0
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720
MAX_TIME = 10.0
MIN_CONFIDENCE = 0.5
WINDOW_NAME = "Security Monitor"

# ==================== INITIALIZATION ====================

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

mp_drawing = mp.solutions.drawing_utils
mp_face = mp.solutions.face_detection

face_detection = mp_face.FaceDetection(
    model_selection=0,
    min_detection_confidence=MIN_CONFIDENCE
)

# Security timer
face_start_time = None
alert_active = False

# FPS calculation
fps_start_time = time.time()
frame_count = 0
fps = 0.0

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

# ==================== MAIN LOOP ====================

while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame_count += 1

    h, w, _ = frame.shape

    # ==================== FACE DETECTION ====================

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_rgb.flags.writeable = False

    results = face_detection.process(frame_rgb)

    frame_rgb.flags.writeable = True

    # ==================== SECURITY LOGIC ====================

    face_detected = bool(results.detections)

    if face_detected:

        if face_start_time is None:
            face_start_time = time.time()
            alert_active = False

        elapsed_time = time.time() - face_start_time

        if elapsed_time >= MAX_TIME:
            alert_active = True

    else:

        face_start_time = None
        alert_active = False
        elapsed_time = 0.0

    # ==================== DRAW FACE DETECTIONS ====================

    if results.detections:

        for detection in results.detections:
            mp_drawing.draw_detection(frame, detection)

    # ==================== FPS ====================

    current_time = time.time()

    if current_time - fps_start_time >= 1.0:

        fps = frame_count / (current_time - fps_start_time)

        frame_count = 0
        fps_start_time = current_time

    # ==================== UI ====================

    face_count = len(results.detections) if results.detections else 0

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Faces: {face_count}",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    if face_detected:

        cv2.putText(
            frame,
            f"Presence: {elapsed_time:.1f} / {MAX_TIME:.1f} sec",
            (20, 110),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    else:

        cv2.putText(
            frame,
            "Presence: No Face",
            (20, 110),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    # ==================== STATUS ====================

    if alert_active:

        cv2.rectangle(
            frame,
            (0, 0),
            (w - 1, h - 1),
            (0, 0, 255),
            8
        )

        cv2.putText(
            frame,
            "SECURITY ALERT",
            (20, 155),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 0, 255),
            3
        )

        cv2.setWindowProperty(
            WINDOW_NAME,
            cv2.WND_PROP_TOPMOST,
            1
        )

    elif face_detected:

        cv2.putText(
            frame,
            "MONITORING",
            (20, 155),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 255),
            2
        )

    else:

        cv2.putText(
            frame,
            "SYSTEM READY",
            (20, 155),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2
        )

    # ==================== DISPLAY ====================

    cv2.imshow(WINDOW_NAME, frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# ==================== CLEANUP ====================

cap.release()
face_detection.close()
cv2.destroyAllWindows()

